# GRPO、CISPO、DAPO、GSPO：从 tensor 到梯度

本 Notebook 直接调用 `trainer/policy_optimization.py`。所有实验使用小 tensor，目的是理解公式、reduction 和梯度，不是训练效果证明。

建议先阅读 `docs/foundation_model_interview/03_POLICY_OPTIMIZATION_FROM_ZERO_TO_MASTERY.md`。

In [ ]:
from pathlib import Path
import inspect
import math
import sys
import torch

repo = Path.cwd().resolve()
if not (repo / 'trainer').exists():
    repo = repo.parent
assert (repo / 'trainer' / 'policy_optimization.py').exists(), '请从仓库根目录或 learning_notebooks 目录运行'
sys.path.insert(0, str(repo))
torch.manual_seed(42)
print('repo =', repo, '| torch =', torch.__version__)

## 1. Group-relative advantage：同一 prompt 内比较

输入顺序必须是每个 prompt 的 G 条回答连续排列。全对或全错组会得到全零 advantage。

In [ ]:
from trainer.policy_optimization import (
    compute_policy_loss, effective_group_mask, group_relative_advantages,
    positive_kl_estimate, soft_overlong_penalty,
)

rewards = torch.tensor([-1., -1., 1., 1.,  -1., -1., -1., -1.])
advantages = group_relative_advantages(rewards, group_size=4)
print('rewards grouped:')
print(rewards.view(-1, 4))
print('advantages grouped:')
print(advantages.view(-1, 4))
assert torch.allclose(advantages[4:], torch.zeros(4))

## 2. 四种目标使用同一批 log-prob

`current` 有梯度，`old` 是 behavior policy，`reference` 是冻结 SFT 锚点。mask=0 的位置不会贡献 loss。

In [ ]:
old = torch.tensor([[-1.05, -1.20, -1.35], [-0.75, -0.85, -0.95]])
reference = torch.tensor([[-1.10, -1.30, -1.50], [-0.80, -0.90, -1.00]])
mask = torch.tensor([[1., 1., 0.], [1., 1., 1.]])
trajectory_advantage = torch.tensor([1., -1.])
base_current = torch.tensor([[-1.00, -1.20, -1.40], [-0.70, -0.80, -0.90]])

rows = []
for algorithm in ['grpo', 'cispo', 'dapo', 'gspo']:
    current = base_current.clone().requires_grad_(True)
    out = compute_policy_loss(
        loss_type=algorithm,
        current_logps=current, old_logps=old, reference_logps=reference,
        advantages=trajectory_advantage, completion_mask=mask, beta=0.01,
    )
    out.loss.backward()
    rows.append({
        'algorithm': algorithm, 'loss': out.loss.item(),
        'ratio_mean': out.ratio_mean.item(),
        'clip_fraction': out.clip_fraction.item(),
        'kl_k3': out.approx_kl.item(),
        'grad_norm': current.grad.norm().item(),
    })
for row in rows:
    print(row)
assert all(math.isfinite(row['loss']) and row['grad_norm'] > 0 for row in rows)

不要横向解释四个 `policy_loss` 的绝对大小：CISPO 使用加权 log-prob，GRPO/DAPO 使用 surrogate，GSPO 又是序列 reduction。应该比较 Accuracy、Reward、KL、长度和稳定性。

## 3. 第一次 policy forward invariant：current=old 时 ratio=1

In [ ]:
current = old.clone().requires_grad_(True)
out = compute_policy_loss(
    loss_type='gspo', current_logps=current, old_logps=old,
    reference_logps=old, advantages=trajectory_advantage, completion_mask=mask,
)
print('ratio_mean =', out.ratio_mean.item(), 'kl_k3 =', out.approx_kl.item())
assert abs(out.ratio_mean.item() - 1.0) < 1e-6
assert abs(out.approx_kl.item()) < 1e-6

## 4. CISPO：裁剪 coefficient，但保留 log-prob 梯度

当 ratio=5.5、`epsilon_high_IS=5` 时，实际上界是 6，因此 coefficient 不会被误裁成 5。

In [ ]:
current = torch.log(torch.tensor([[5.5]])).requires_grad_(True)
out = compute_policy_loss(
    loss_type='cispo', current_logps=current, old_logps=torch.zeros_like(current),
    advantages=torch.ones(1), completion_mask=torch.ones_like(current),
    cispo_epsilon_high=5.0,
)
out.loss.backward()
print('ratio =', out.ratio_mean.item(), 'dL/dlogp =', current.grad.item())
assert abs(current.grad.item() + 5.5) < 1e-5

## 5. DAPO Dynamic Sampling：只保留部分成功组

In [ ]:
task_success = torch.tensor([0, 1, 0, 1,  1, 1, 1, 1,  0, 0, 0, 0], dtype=torch.bool)
keep = effective_group_mask(task_success, group_size=4)
print('success grouped:')
print(task_success.view(-1, 4).int())
print('keep prompt groups:', keep.tolist())
assert keep.tolist() == [True, False, False]

## 6. DAPO Soft Overlong：在长度上限前线性惩罚

In [ ]:
lengths = torch.tensor([70, 80, 90, 100, 110])
penalty = soft_overlong_penalty(lengths, max_length=100, cache_length=20)
for length, value in zip(lengths.tolist(), penalty.tolist()):
    print(f'length={length:3d} penalty={value:+.2f}')
assert penalty.tolist() == [0.0, -0.0, -0.5, -1.0, -1.0]

## 7. Sequence-first 与 Token-level reduction

两条序列长度为 1 和 3。sequence-first 让两条各占 50%；global token mean 让四个 token 各占 25%。

In [ ]:
token_losses = torch.tensor([[2., 0., 0.], [1., 1., 1.]])
reduction_mask = torch.tensor([[1., 0., 0.], [1., 1., 1.]])
sequence_means = (token_losses * reduction_mask).sum(1) / reduction_mask.sum(1)
sequence_first = sequence_means.mean()
token_mean = (token_losses * reduction_mask).sum() / reduction_mask.sum()
print('sequence means:', sequence_means.tolist())
print('sequence-first =', sequence_first.item(), 'global token mean =', token_mean.item())
assert sequence_first.item() != token_mean.item()

## 8. GSPO：sequence ratio 是 token ratio 的几何平均

In [ ]:
token_ratios = torch.tensor([0.5, 2.0])
arithmetic = token_ratios.mean()
geometric = torch.exp(torch.log(token_ratios).mean())
print('arithmetic mean =', arithmetic.item())
print('GSPO geometric mean =', geometric.item())
assert abs(geometric.item() - 1.0) < 1e-6

## 9. k3 KL estimator 是逐点非负的

In [ ]:
current = torch.tensor([[-1.0, -2.0, -3.0]])
ref = torch.tensor([[-2.0, -1.0, -3.0]])
k3 = positive_kl_estimate(current, ref)
print(k3)
assert torch.all(k3 >= 0)

## 10. 查看真实统一目标源码

In [ ]:
source = inspect.getsource(compute_policy_loss)
print('compute_policy_loss lines =', len(source.splitlines()))
for keyword in ['cispo', 'grpo', 'dapo', 'gspo', 'seq_log_ratio', 'policy_loss']:
    print(keyword, '=>', source.count(keyword))

## 11. 下一步实际运行

1. 运行 `python -m unittest tests.test_policy_optimization -v`。
2. 按主教程分别运行 GRPO、CISPO、DAPO、GSPO 的小数据 debug。
3. 检查第一次 forward 的 `rollout_logprob_mae≈0`、`rollout_ratio_mean≈1`。
4. 做 DAPO 逐项消融和 GSPO clip grid。
5. 最后才运行四算法 × 三训练 seed 的正式实验，并同时报告 optimizer 与 environment budget。